<a href="https://colab.research.google.com/github/ananthakrishnanm010/Helmet-and-Rider-Safety-Violation-Detection/blob/main/HelmetSafety_yolov8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing Ultralytics package
-Ultralytics is a Python package, which gives you tools to train and use YOLO models.

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.1 MB/s eta 0:00:00


In [2]:
!pip install -q -U kagglehub


# Dataset Reading

In [6]:

import os
import yaml
import kagglehub

# Download the public Kaggle dataset
dataset_root = kagglehub.dataset_download(
    "febinnemmanuel/helmet-safety-yolov8-dataset"
)

print("Downloaded to:", dataset_root)

# Inspect the downloaded directory
print("Contents:", os.listdir(dataset_root))

# Find data.yaml anywhere inside the downloaded dataset
DATASET_PATH = None

for root, dirs, files in os.walk(dataset_root):
    if "data.yaml" in files:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "data.yaml was not found inside the downloaded dataset."
    )

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

print("DATASET_PATH:", DATASET_PATH)
print("YAML_PATH:", YAML_PATH)
print("Dataset contents:", os.listdir(DATASET_PATH))


Using Colab cache for faster access to the 'helmet-safety-yolov8-dataset' dataset.
Downloaded to: /kaggle/input/helmet-safety-yolov8-dataset
Contents: ['content']
DATASET_PATH: /kaggle/input/helmet-safety-yolov8-dataset/content/helmet_yolo
YAML_PATH: /kaggle/input/helmet-safety-yolov8-dataset/content/helmet_yolo/data.yaml
Dataset contents: ['labels', 'data.yaml', 'images']


In [14]:
import os
import shutil

SOURCE_PATH = "/kaggle/input/helmet-safety-yolov8-dataset/content/helmet_yolo"
DATASET_PATH = "/content/helmet_yolo"

if not os.path.exists(DATASET_PATH):
    shutil.copytree(SOURCE_PATH, DATASET_PATH)

print("Dataset copied to:", DATASET_PATH)
print("Contents:", os.listdir(DATASET_PATH))

Dataset copied to: /content/helmet_yolo
Contents: ['labels', 'images', 'data.yaml']


In [15]:
import yaml
import os

YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

data["path"] = DATASET_PATH

with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(open(YAML_PATH).read())

path: /content/helmet_yolo
train: images/train
val: images/val
test: images/test
names:
  0: With Helmet
  1: Without Helmet



In [16]:
for split in ["train", "val", "test"]:
    folder = os.path.join(DATASET_PATH, "images", split)
    print(split, "exists:", os.path.isdir(folder))

    if os.path.isdir(folder):
        print("Number of files:", len(os.listdir(folder)))

train exists: True
Number of files: 534
val exists: True
Number of files: 153
test exists: True
Number of files: 77


In [17]:
import os
import yaml

print("Dataset path:", DATASET_PATH)
print("\nDataset YAML:")

with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

print(data)

for split in ["train", "val", "test"]:
    image_dir = os.path.join(DATASET_PATH, "images", split)
    label_dir = os.path.join(DATASET_PATH, "labels", split)

    print(f"\n{split.upper()}")
    print("Images:", len(os.listdir(image_dir)) if os.path.isdir(image_dir) else "MISSING")
    print("Labels:", len(os.listdir(label_dir)) if os.path.isdir(label_dir) else "MISSING")

Dataset path: /content/helmet_yolo

Dataset YAML:
{'path': '/content/helmet_yolo', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'With Helmet', 1: 'Without Helmet'}}

TRAIN
Images: 534
Labels: 534

VAL
Images: 153
Labels: 153

TEST
Images: 77
Labels: 77


# Yolov8n

In [18]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

# Model Baseline Training

In [19]:
results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/helmet_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optimize=False, opti

In [20]:
metrics = model.val()
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3697.8±863.2 MB/s, size: 321.0 KB)
val: Scanning /content/helmet_yolo/labels/val.cache... 153 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 153/153 45.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.4it/s 4.2s
                   all        153        251      0.746      0.808      0.813      0.508
           With Helmet        115        174      0.829      0.851      0.912      0.614
        Without Helmet         47         77      0.663      0.766      0.714      0.403
Speed: 3.2ms preprocess, 6.8ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /content/runs/detect/val
mAP@50: 0.8128726160556803
mAP@50-95: 0.5080276471167235
Precision: 0.7457146658785831
Recall: 0.80

### Model Evaluation on the Test Set

In [21]:
# Load the best weights from our baseline training run
best_model_path = '/content/runs/detect/train-2/weights/best.pt'
best_model = YOLO(best_model_path)

# Evaluate on the test split by setting split='test'
test_metrics = best_model.val(split='test')

print("\n--- Test Dataset Performance Metrics ---")
print(f"mAP@50: {test_metrics.box.map50:.4f}")
print(f"mAP@50-95: {test_metrics.box.map:.4f}")
print(f"Precision (mp): {test_metrics.box.mp:.4f}")
print(f"Recall (mr): {test_metrics.box.mr:.4f}")

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3730.6±640.5 MB/s, size: 561.2 KB)
val: Scanning /content/helmet_yolo/labels/test... 77 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 77/77 655.0it/s 0.1s
val: New cache created: /content/helmet_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.2s
                   all         77        156      0.841      0.791      0.876      0.521
           With Helmet         49         89      0.859      0.888      0.916      0.603
        Without Helmet         33         67      0.823      0.694      0.836      0.438
Speed: 7.1ms preprocess, 10.8ms inference, 0.0ms loss, 4.3ms postprocess per image
Results saved to /content/runs/detect/val-2

--- Test Dataset Performance Metrics ---
m